# pRESTO merge пар ридов — мышь (ERP003950)

Оптимизированная сборка пар ридов pRESTO для `ERP003950`. Отличия от exploratory-ноутбука:
- явный `--nproc` вместо дефолтного pRESTO `56`;
- verbose per-read `--log` по умолчанию выключен (может вырасти на >1GB на sample и тормозить I/O);
- опциональная запись `--failed` reads по умолчанию выключена, чтобы меньше писать на диск;
- пишет накопительный `assembly_qc.tsv` после каждого sample;
- QC здесь не считается — после сборки используй `qc.ipynb` с `label='merged'`.


In [ ]:
import os, sys, sysconfig
_ENV_CANDIDATES = [
    "/data/user/epishkin/venvs/presto_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)
print(f"Using merge environment: {_CONDA_ENV}")


In [ ]:
import gzip, shutil, subprocess, time, csv
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
LABEL = "pr_trimmed"
NPROC = 16
WRITE_VERBOSE_LOG = False
WRITE_FAILED_READS = False
GZIP_OUTPUT = True
FORCE = False


In [ ]:
def fastq_dir(volume, dataset, label):
    vol = Path(volume)
    mapping = {
        "trimmed": vol / "results" / dataset / "trimmed" / "fastq",
        "pr_trimmed": vol / "results" / dataset / "pr_trimmed" / "fastq",
    }
    if label not in mapping:
        raise ValueError(f"Unsupported label: {label}")
    return mapping[label]

def discover_pairs(input_dir):
    input_dir = Path(input_dir)
    patterns = [
        ("*_1.pr.fastq.gz", "_1.pr.fastq.gz", "_2.pr.fastq.gz"),
        ("*_1.trim.fastq.gz", "_1.trim.fastq.gz", "_2.trim.fastq.gz"),
    ]
    pairs, seen = [], set()
    for glob_pat, r1_suffix, r2_suffix in patterns:
        for r1 in sorted(input_dir.glob(glob_pat)):
            sample = r1.name[:-len(r1_suffix)]
            if sample in seen:
                continue
            r2 = input_dir / f"{sample}{r2_suffix}"
            if not r2.exists():
                raise FileNotFoundError(f"Missing mate for {r1}: expected {r2}")
            pairs.append((sample, r1, r2))
            seen.add(sample)
    return pairs

def count_fastq_records(path):
    path = Path(path)
    if not path.exists():
        return 0
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt") as handle:
        return sum(1 for _ in handle) // 4

def output_paths(out_fastq, sample):
    suffix = ".fastq.gz" if GZIP_OUTPUT else ".fastq"
    return {
        "pass": out_fastq / f"{sample}_assemble-pass{suffix}",
        "fail1": out_fastq / f"{sample}-1_assemble-fail{suffix}",
        "fail2": out_fastq / f"{sample}-2_assemble-fail{suffix}",
    }


In [ ]:
def run_merge_pairs_optimized(volume=VOLUME, dataset=DATASET, label=LABEL, nproc=NPROC,
                              write_verbose_log=WRITE_VERBOSE_LOG,
                              write_failed_reads=WRITE_FAILED_READS,
                              gzip_output=GZIP_OUTPUT,
                              force=FORCE):
    if not shutil.which("AssemblePairs.py"):
        raise RuntimeError("AssemblePairs.py not found. Use BCR Pipeline kernel or run setup_vm_conda.sh.")

    vol = Path(volume)
    input_dir = fastq_dir(vol, dataset, label)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input FASTQ directory not found: {input_dir}")

    out_base = vol / "results" / dataset / "merged"
    out_fastq = out_base / "fastq"
    out_logs = out_base / "logs"
    out_qc = out_base / "qc"
    for d in (out_fastq, out_logs, out_qc):
        d.mkdir(parents=True, exist_ok=True)

    pairs = discover_pairs(input_dir)
    if not pairs:
        raise FileNotFoundError(f"No paired FASTQ files found in {input_dir}")

    print(f"[merge_pairs_optimized] dataset={dataset} label={label}")
    print(f"[merge_pairs_optimized] input={input_dir}")
    print(f"[merge_pairs_optimized] pairs={len(pairs)} nproc={nproc}")
    print(f"[merge_pairs_optimized] verbose_log={write_verbose_log} failed_reads={write_failed_reads} gzip_output={gzip_output}")

    rows = []
    for idx, (sample, r1, r2) in enumerate(pairs, start=1):
        paths = output_paths(out_fastq, sample)
        pass_fastq = paths["pass"]
        fail1_fastq = paths["fail1"]
        fail2_fastq = paths["fail2"]
        stdout_path = out_logs / f"{sample}.stdout.txt"
        stderr_path = out_logs / f"{sample}.stderr.txt"
        log_path = out_logs / f"{sample}.assemble.log"

        r1_count = count_fastq_records(r1)
        r2_count = count_fastq_records(r2)
        if r1_count != r2_count:
            raise RuntimeError(f"Input counts differ for {sample}: {r1_count} vs {r2_count}")

        if pass_fastq.exists() and not force:
            print(f"  [{idx}/{len(pairs)}] [skip] {sample}: {pass_fastq.name} exists")
            status = "skipped"
            elapsed = 0.0
        else:
            cmd = [
                "AssemblePairs.py", "align",
                "-1", str(r1),
                "-2", str(r2),
                "--coord", "illumina",
                "--rc", "tail",
                "--outname", sample,
                "--outdir", str(out_fastq),
                "--nproc", str(nproc),
            ]
            if gzip_output:
                cmd.append("--gzip-output")
            if write_failed_reads:
                cmd.append("--failed")
            if write_verbose_log:
                cmd += ["--log", str(log_path)]

            print(f"  [{idx}/{len(pairs)}] [run] {sample}")
            t0 = time.time()
            # heartbeat, чтобы длинный AssemblePairs не выглядел "зависшим"
            with open(stdout_path, "w") as stdout_handle, open(stderr_path, "w") as stderr_handle:
                proc = subprocess.Popen(cmd, stdout=stdout_handle, stderr=stderr_handle, text=True)
                print(f"    pid={proc.pid} stdout={stdout_path.name} stderr={stderr_path.name}")
                heartbeat = 30
                while True:
                    rc = proc.poll()
                    if rc is not None:
                        break
                    elapsed_live = time.time() - t0
                    print(f"    still running: pid={proc.pid} elapsed={elapsed_live/60:.1f} min", flush=True)
                    time.sleep(heartbeat)
            elapsed = time.time() - t0
            if rc != 0:
                raise RuntimeError(f"AssemblePairs failed for {sample} with exit code {rc}; see {stderr_path}")
            status = "done"

        n_pass = count_fastq_records(pass_fastq)
        n_fail = count_fastq_records(fail1_fastq) + count_fastq_records(fail2_fastq)
        total = n_pass + n_fail
        rate = n_pass / total if total else 0.0
        rows.append({
            "sample": sample,
            "status": status,
            "input_r1_records": str(r1_count),
            "input_r2_records": str(r2_count),
            "elapsed_sec": f"{elapsed:.1f}",
            "assembled_pairs": str(n_pass),
            "failed_reads": str(n_fail),
            "total_seen_records": str(total),
            "merge_rate_from_outputs": f"{rate:.6f}",
            "pass_fastq": str(pass_fastq),
            "fail1_fastq": str(fail1_fastq),
            "fail2_fastq": str(fail2_fastq),
            "stdout": str(stdout_path),
            "stderr": str(stderr_path),
            "verbose_log": str(log_path) if write_verbose_log else "",
        })
        qc_path = out_qc / "assembly_qc.tsv"
        with open(qc_path, "w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()), delimiter="\t")
            writer.writeheader()
            writer.writerows(rows)
        print(f"    pass={n_pass:,} fail_records={n_fail:,} elapsed={elapsed/60:.1f} min")

    total_pass = sum(int(r["assembled_pairs"]) for r in rows)
    total_fail = sum(int(r["failed_reads"]) for r in rows)
    print(f"[merge_pairs_optimized] qc: {out_qc / 'assembly_qc.tsv'}")
    print(f"[merge_pairs_optimized] assembled_pairs={total_pass:,}")
    print(f"[merge_pairs_optimized] failed_records={total_fail:,}")


### Запуск


In [ ]:
run_merge_pairs_optimized()
